In [ ]:
from pathlib import Path
import re
import pandas as pd
from difflib import SequenceMatcher

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
REPO_ROOT = PROJECT_ROOT

DATA_PROCESSED = REPO_ROOT / "data" / "processed"
PERSONAL_DIR = DATA_PROCESSED / "personal_atlas"

league_path = PERSONAL_DIR / "bts_song_league.parquet"
atlas_path = DATA_PROCESSED / "song_embeddings.parquet"

bts_song_league = pd.read_parquet(league_path)
atlas_df = pd.read_parquet(atlas_path)

bts_song_league.shape, atlas_df.shape


In [ ]:
def normalize_title(title: str) -> str:
    if pd.isna(title):
        return ""

    title = str(title).lower()

    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"\[.*?\]", "", title)

    remove_terms = [
        "remastered",
        "remaster",
        "explicit",
        "clean version",
        "instrumental",
        "karaoke",
    ]

    for term in remove_terms:
        title = title.replace(term, "")

    title = title.replace("&", "and")
    title = re.sub(r"[^a-z0-9가-힣ぁ-んァ-ン一-龯\s]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title


bts_song_league["normalized_history_title"] = bts_song_league[
    "master_metadata_track_name"
].apply(normalize_title)

atlas_df["normalized_atlas_title"] = atlas_df["track_name"].apply(normalize_title)

In [ ]:
exact_matches = bts_song_league.merge(
    atlas_df[["track_id", "track_name", "album_name", "release_year", "normalized_atlas_title"]],
    left_on="normalized_history_title",
    right_on="normalized_atlas_title",
    how="left",
)

exact_matches["match_type"] = exact_matches["track_id"].notna().map(
    {True: "exact", False: None}
)

exact_matches["track_id"].notna().mean()

In [ ]:
atlas_titles = atlas_df[["track_id", "track_name", "album_name", "release_year", "normalized_atlas_title"]].drop_duplicates()

def best_fuzzy_match(query: str, choices: pd.DataFrame) -> dict:
    best = None
    best_score = 0

    for _, row in choices.iterrows():
        candidate = row["normalized_atlas_title"]
        score = SequenceMatcher(None, query, candidate).ratio()

        if score > best_score:
            best_score = score
            best = row

    if best is None:
        return {
            "matched_track_id": None,
            "matched_track_name": None,
            "matched_album_name": None,
            "matched_release_year": None,
            "fuzzy_score": 0,
        }

    return {
        "matched_track_id": best["track_id"],
        "matched_track_name": best["track_name"],
        "matched_album_name": best["album_name"],
        "matched_release_year": best["release_year"],
        "fuzzy_score": best_score,
    }


unmatched = exact_matches[exact_matches["track_id"].isna()].copy()

fuzzy_rows = []
for _, row in unmatched.iterrows():
    result = best_fuzzy_match(row["normalized_history_title"], atlas_titles)
    fuzzy_rows.append(result)

fuzzy_df = pd.DataFrame(fuzzy_rows, index=unmatched.index)

for col in fuzzy_df.columns:
    unmatched[col] = fuzzy_df[col]

unmatched["match_type"] = unmatched["fuzzy_score"].apply(
    lambda x: "fuzzy" if x >= 0.86 else "unmatched"
)

unmatched[
    [
        "master_metadata_album_artist_name",
        "master_metadata_track_name",
        "hours_played",
        "matched_track_name",
        "fuzzy_score",
        "match_type",
    ]
].sort_values("hours_played", ascending=False).head(50)

In [ ]:
exact_ready = exact_matches[exact_matches["track_id"].notna()].copy()

exact_ready["matched_track_id"] = exact_ready["track_id"]
exact_ready["matched_track_name"] = exact_ready["track_name"]
exact_ready["matched_album_name"] = exact_ready["album_name"]
exact_ready["matched_release_year"] = exact_ready["release_year"]
exact_ready["fuzzy_score"] = 1.0

accepted_fuzzy = unmatched[unmatched["match_type"] == "fuzzy"].copy()

matched_history = pd.concat(
    [
        exact_ready[
            [
                "master_metadata_album_artist_name",
                "master_metadata_track_name",
                "master_metadata_album_album_name",
                "plays",
                "hours_played",
                "total_ms",
                "first_play",
                "last_play",
                "matched_track_id",
                "matched_track_name",
                "matched_album_name",
                "matched_release_year",
                "match_type",
                "fuzzy_score",
            ]
        ],
        accepted_fuzzy[
            [
                "master_metadata_album_artist_name",
                "master_metadata_track_name",
                "master_metadata_album_album_name",
                "plays",
                "hours_played",
                "total_ms",
                "first_play",
                "last_play",
                "matched_track_id",
                "matched_track_name",
                "matched_album_name",
                "matched_release_year",
                "match_type",
                "fuzzy_score",
            ]
        ],
    ],
    ignore_index=True,
)

matched_history.shape

In [ ]:
personal_track_stats = (
    matched_history
    .groupby("matched_track_id", dropna=False)
    .agg(
        personal_plays=("plays", "sum"),
        personal_hours=("hours_played", "sum"),
        personal_total_ms=("total_ms", "sum"),
        personal_first_play=("first_play", "min"),
        personal_last_play=("last_play", "max"),
        source_history_titles=("master_metadata_track_name", lambda x: " | ".join(sorted(set(x)))),
        match_types=("match_type", lambda x: " | ".join(sorted(set(x)))),
        min_fuzzy_score=("fuzzy_score", "min"),
    )
    .reset_index()
    .rename(columns={"matched_track_id": "track_id"})
)

In [ ]:
def mastery_level(hours: float) -> int:
    if hours <= 0:
        return 0
    if hours < 0.5:
        return 1
    if hours < 1:
        return 2
    if hours < 2:
        return 3
    if hours < 5:
        return 4
    if hours < 10:
        return 5
    if hours < 20:
        return 6
    if hours < 40:
        return 7
    if hours < 75:
        return 8
    if hours < 150:
        return 9
    return 10


personal_track_stats["personal_rank"] = (
    personal_track_stats["personal_hours"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

personal_track_stats["mastery_level"] = personal_track_stats["personal_hours"].apply(mastery_level)

In [ ]:
personal_overlay = atlas_df.merge(
    personal_track_stats,
    on="track_id",
    how="left",
)

personal_overlay["personal_plays"] = personal_overlay["personal_plays"].fillna(0).astype(int)
personal_overlay["personal_hours"] = personal_overlay["personal_hours"].fillna(0)
personal_overlay["personal_total_ms"] = personal_overlay["personal_total_ms"].fillna(0).astype(int)
personal_overlay["mastery_level"] = personal_overlay["mastery_level"].fillna(0).astype(int)
personal_overlay["has_listening_history"] = personal_overlay["personal_plays"] > 0

personal_overlay[
    ["track_name", "album_name", "personal_hours", "personal_plays", "personal_rank", "mastery_level"]
].sort_values("personal_hours", ascending=False).head(30)

In [ ]:
personal_overlay_path = PERSONAL_DIR / "personal_atlas_overlay.parquet"
matched_history_path = PERSONAL_DIR / "matched_listening_history.parquet"

personal_overlay.to_parquet(personal_overlay_path, index=False)
matched_history.to_parquet(matched_history_path, index=False)

personal_overlay_path, matched_history_path

In [ ]:
summary = {
    "history_songs": len(bts_song_league),
    "matched_history_rows": len(matched_history),
    "atlas_tracks": len(atlas_df),
    "atlas_tracks_with_history": personal_overlay["has_listening_history"].sum(),
    "total_personal_hours_matched": personal_overlay["personal_hours"].sum(),
}

summary

In [ ]:
personal_overlay[
    ["track_name", "album_name", "personal_hours", "personal_plays", "mastery_level"]
].sort_values("personal_hours", ascending=False).head(30)

In [ ]:
matched_history.query("match_type == 'fuzzy'")[
    ["master_metadata_track_name", "matched_track_name", "fuzzy_score", "hours_played"]
].sort_values("fuzzy_score").head(30)

## Canonical Matching

In [ ]:
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
PERSONAL_DIR = DATA_PROCESSED / "personal_atlas"

atlas_df = pd.read_parquet(DATA_PROCESSED / "song_embeddings.parquet")
league = pd.read_parquet(PERSONAL_DIR / "bts_song_league.parquet")

print(atlas_df.columns.tolist()[:30])
print(atlas_df.shape)
print(league.shape)

In [ ]:
[col for col in atlas_df.columns if "canon" in col.lower() or "title" in col.lower() or "name" in col.lower()]

In [ ]:
atlas_df[["track_name", "album_name"]].head(20)

In [ ]:
def canonicalize_title(title):
    if pd.isna(title):
        return ""

    title = str(title).lower()

    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"\[.*?\]", "", title)

    remove_phrases = [
        "full length edition",
        "steve aoki remix",
        "rocking vibe mix",
        "urban mix",
        "remix",
        "remastered",
        "remaster",
        "live",
        "feat desiigner",
        "feat.",
        "feat",
    ]

    for phrase in remove_phrases:
        title = title.replace(phrase, "")

    title = title.replace("&", "and")
    title = re.sub(r"[^a-z0-9가-힣ぁ-んァ-ン一-龯\s]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title

atlas_df["canonical_key"] = atlas_df["track_name"].apply(canonicalize_title)
league["history_key"] = league["master_metadata_track_name"].apply(canonicalize_title)

In [ ]:
canonical_atlas = (
    atlas_df
    .groupby("canonical_key", as_index=False)
    .agg(
        canonical_title=("track_name", "first"),
        representative_album=("album_name", "first"),
        version_count=("track_name", "count"),
        atlas_versions=("track_name", lambda x: sorted(set(x))),
        atlas_albums=("album_name", lambda x: sorted(set(x))),
    )
)

canonical_atlas.sort_values("version_count", ascending=False).head(30)

In [ ]:
matched = league.merge(
    canonical_atlas,
    left_on="history_key",
    right_on="canonical_key",
    how="left",
)

matched["match_type"] = matched["canonical_title"].notna().map({
    True: "exact",
    False: "unmatched"
})

matched["match_type"].value_counts()

In [ ]:
matched[matched["match_type"] == "unmatched"][
    [
        "master_metadata_album_artist_name",
        "master_metadata_track_name",
        "master_metadata_album_album_name",
        "hours_played",
        "plays",
        "history_key",
    ]
].sort_values("hours_played", ascending=False).head(50)


In [ ]:
print(f"History songs: {len(league)}")
print(f"Matched: {matched['canonical_title'].notna().sum()}")
print(f"Coverage: {matched['canonical_title'].notna().mean():.1%}")

print("\nTop unmatched artists:")
print(
    matched.loc[matched["canonical_title"].isna(), "master_metadata_album_artist_name"]
    .value_counts()
)

In [ ]:
matched[
    (matched["canonical_title"].isna()) &
    (matched["master_metadata_album_artist_name"] == "BTS")
][
    [
        "master_metadata_track_name",
        "master_metadata_album_album_name",
        "hours_played"
    ]
].sort_values("hours_played", ascending=False)

In [ ]:
EXCLUDED_TRACK_TYPES = {
    "skit",
    "interlude",
}

EXCLUDED_TRACKS = {
    "2.0 (Live at Pier 17) | Presented by Spotify",
}

In [ ]:
def should_ignore(track_name):
    lower = track_name.lower()

    if "skit" in lower:
        return True

    if "interlude" in lower:
        return True

    if track_name in EXCLUDED_TRACKS:
        return True

    return False

# Conclusion

## Summary

This notebook successfully links Spotify Extended Streaming History with the current BTS Song Atlas dataset.

### Results

- History songs: 344
- Matched songs: 247
- Coverage: 71.8%

### Findings

Most unmatched songs are **not matching errors**.

They primarily belong to:

- Solo discographies (planned for v1.1)
- BTS songs not yet included in the atlas
- Intentional exclusions such as skits and special live releases

The matching pipeline is therefore considered production-ready for the current scope.

Future atlas expansions will automatically improve coverage without requiring changes to the matching logic.

## Outputs

Generated:

- spotify_history_clean.parquet
- bts_listening_history.parquet
- bts_song_league.parquet
- bts_album_league.parquet
- matched_listening_history.parquet
- personal_atlas_overlay.parquet

In [ ]:
from pathlib import Path

OUT_DIR = PROJECT_ROOT / "data" / "processed" / "personal_atlas"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Final matched listening history
matched.to_parquet(
    OUT_DIR / "matched_listening_history.parquet",
    index=False,
)

# One row per atlas song with personal stats
personal_overlay.to_parquet(
    OUT_DIR / "personal_atlas_overlay.parquet",
    index=False,
)

# Canonical atlas lookup (useful for future notebooks)
canonical_atlas.to_parquet(
    OUT_DIR / "canonical_artist_lookup.parquet",
    index=False,
)

print("✅ Saved:")
print(f"  • {OUT_DIR / 'matched_listening_history.parquet'}")
print(f"  • {OUT_DIR / 'personal_atlas_overlay.parquet'}")
print(f"  • {OUT_DIR / 'canonical_song_lookup.parquet'}")